In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
df.head()

Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [2]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (6362620, 11)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Data types:
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Duplicate rows: 0


In [3]:
fraud_counts = df["isFraud"].value_counts()

print(fraud_counts)

print("\nFraud percentage:")
print((df["isFraud"].value_counts(normalize=True) * 100).round(4))

isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud percentage:
isFraud
0    99.8709
1     0.1291
Name: proportion, dtype: float64


In [4]:
transaction_summary = (
    df.groupby("type")
      .agg(
          transactions=("isFraud", "size"),
          fraud_count=("isFraud", "sum"),
          fraud_rate=("isFraud", "mean")
      )
      .sort_values("fraud_rate", ascending=False)
)

transaction_summary["fraud_rate"] *= 100

transaction_summary

,transactions,fraud_count,fraud_rate
type,,,
TRANSFER,532909,4097,0.768799
CASH_OUT,2237500,4116,0.183955
CASH_IN,1399284,0,0.000000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000


In [5]:
df.groupby("type")["isFraud"].sum().sort_values(ascending=False)

type
CASH_OUT    4116
TRANSFER    4097
CASH_IN        0
DEBIT          0
PAYMENT        0
Name: isFraud, dtype: int64

In [6]:
transaction_summary = (
    df.groupby("type")
      .agg(
          transactions=("isFraud", "size"),
          fraud_count=("isFraud", "sum"),
          fraud_rate=("isFraud", "mean")
      )
)

transaction_summary["fraud_rate_pct"] = (
    transaction_summary["fraud_rate"] * 100
)

transaction_summary = transaction_summary.sort_values(
    "fraud_rate_pct",
    ascending=False
)

transaction_summary

,transactions,fraud_count,fraud_rate,fraud_rate_pct
type,,,,
TRANSFER,532909,4097,0.007688,0.768799
CASH_OUT,2237500,4116,0.001840,0.183955
CASH_IN,1399284,0,0.000000,0.000000
DEBIT,41432,0,0.000000,0.000000
PAYMENT,2151495,0,0.000000,0.000000


In [7]:
pd.crosstab(
    df["type"],
    df["isFraud"],
    normalize="index"
).mul(100).round(4)

isFraud,0,1
type,,
CASH_IN,100.0000,0.0000
CASH_OUT,99.8160,0.1840
DEBIT,100.0000,0.0000
PAYMENT,100.0000,0.0000
TRANSFER,99.2312,0.7688


In [8]:
df.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00


In [9]:
df.groupby("type")["amount"].agg(
    ["count", "mean", "median", "min", "max"]
).sort_values("mean", ascending=False)

,count,mean,median,min,max
type,,,,,
TRANSFER,532909,910647.009645,486308.390,2.60,92445516.64
CASH_OUT,2237500,176273.964346,147072.185,0.00,10000000.00
CASH_IN,1399284,168920.242004,143427.710,0.04,1915267.90
PAYMENT,2151495,13057.604660,9482.190,0.02,238637.98
DEBIT,41432,5483.665314,3048.990,0.55,569077.51


In [10]:
df["orig_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

df["dest_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

df["orig_balance_error"] = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
)

df["dest_balance_error"] = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
)

In [11]:
df.groupby("isFraud")[
    [
        "orig_balance_change",
        "dest_balance_change",
        "orig_balance_error",
        "dest_balance_error"
    ]
].mean()

,orig_balance_change,dest_balance_change,orig_balance_error,dest_balance_error
isFraud,,,,
0,-2.314152e+04,123504.809994,-201338.558109,54692.231734
1,1.457275e+06,735457.998071,-10692.325265,732509.301069


In [13]:
df["orig_balance_diff"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

df["orig_amount_gap"] = (
    df["orig_balance_diff"] - df["amount"]
)

df.groupby("isFraud")["orig_amount_gap"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,-201338.558109,606928.890826,-92445516.64,-249953.43,-69049.31,-3034.305,1.000000e-02
1,8213.0,-10692.325265,265146.131130,-10000000.00,0.00,0.00,0.000,3.725290e-09


In [14]:
df["dest_balance_diff"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

df["dest_amount_gap"] = (
    df["dest_balance_diff"] - df["amount"]
)

df.groupby("isFraud")["dest_amount_gap"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,-54692.231734,4.360026e+05,-13191233.98,-29259.805,-3500.68,0.0,75885725.63
1,8213.0,-732509.301069,1.867748e+06,-10000000.00,-442722.010,-2231.46,0.0,8875516.29


In [15]:
df["orig_balance_to_amount"] = (
    df["oldbalanceOrg"] / (df["amount"] + 1)
)

df.groupby("isFraud")["orig_balance_to_amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,47.748534,3301.243037,0.0,0.000000,0.153455,4.283673,3.711165e+06
1,8213.0,1.016783,0.353732,0.0,0.999992,0.999998,0.999999,2.605976e+01


In [16]:
df["dest_balance_to_amount"] = (
    df["oldbalanceDest"] / (df["amount"] + 1)
)

df.groupby("isFraud")["dest_balance_to_amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,41.565413,3516.471185,0.0,0.0,1.09263,6.232451,3.898645e+06
1,8213.0,1780.246554,95731.996416,0.0,0.0,0.00000,0.225974,7.970767e+06


In [17]:
print("Number of time steps:", df["step"].nunique())

print(
    "Step range:",
    df["step"].min(),
    "→",
    df["step"].max()
)

Number of time steps: 743
Step range: 1 → 743


In [18]:
fraud_by_step = (
    df.groupby("step")["isFraud"]
      .agg(
          transactions="size",
          fraud_count="sum"
      )
)

fraud_by_step["fraud_rate"] = (
    fraud_by_step["fraud_count"] /
    fraud_by_step["transactions"]
)

fraud_by_step.sort_values(
    "fraud_rate",
    ascending=False
).head(20)

,transactions,fraud_count,fraud_rate
step,,,
743,8,8,1.0
533,12,12,1.0
269,14,14,1.0
268,6,6,1.0
568,14,14,1.0
572,18,18,1.0
575,8,8,1.0
579,12,12,1.0
581,10,10,1.0


In [19]:
fraud_by_step[fraud_by_step["fraud_count"] > 0].head(20)

,transactions,fraud_count,fraud_rate
step,,,
1,2708,16,0.005908
2,1014,8,0.007890
3,552,4,0.007246
4,565,10,0.017699
5,665,6,0.009023
6,1660,22,0.013253
7,6837,12,0.001755
8,21097,12,0.000569
9,37628,19,0.000505
